In [277]:
# !pip install rectools[lightfm] matplotlib seaborn polars

In [278]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import rectools
from rectools import Columns

In [279]:
# # Считываем датасеты

# df_users = pd.read_csv('data_original/data_original/users.csv')
# df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
# df_items = pd.read_csv('data_original/data_original/items.csv')

In [280]:
df_users = pd.read_csv('KION_DATASET/data_original/users.csv')
df_interactions = pd.read_csv('KION_DATASET/interactions.csv')
df_items = pd.read_csv('KION_DATASET/data_original/items.csv')

In [281]:
df_users = df_users.drop(df_users.index[-1])
df_interactions = df_interactions.drop(df_interactions.index[-1])
df_items = df_items.drop(df_items.index[-1])

In [282]:
df_users.head()

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [283]:
df_interactions.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [284]:
df_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15962 entries, 0 to 15961
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15962 non-null  int64  
 1   content_type  15962 non-null  object 
 2   title         15962 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15864 non-null  float64
 5   genres        15962 non-null  object 
 6   countries     15925 non-null  object 
 7   for_kids      565 non-null    float64
 8   age_rating    15960 non-null  float64
 9   studios       1065 non-null   object 
 10  directors     14453 non-null  object 
 11  actors        13343 non-null  object 
 12  description   15960 non-null  object 
 13  keywords      15539 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.7+ MB


In [285]:
# df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [286]:
print('Зарегистрированных пользователей: ',  len(df_users['user_id'].unique()), 
      'Пользователей с просмотрами: ', len(df_interactions['user_id'].unique()), sep='\n')

Зарегистрированных пользователей: 
840196
Пользователей с просмотрами: 
567588


In [287]:
print('Всего фильмов: ',  len(df_items['item_id'].unique()), 
      'Фильмы с просмотрами: ', len(df_interactions['item_id'].unique()), sep='\n')

Всего фильмов: 
15962
Фильмы с просмотрами: 
12693


In [288]:
df_inter = df_interactions.drop(['total_dur'], axis=1, inplace=False)

# format_check = check_date_with_formats(df_inter_for_ds, 'last_watch_dt')
# print("Результаты проверки форматов:")
# for key, value in format_check.items():
#     print(f"{key}: {value}")
     

print(df_inter["item_id"].count())


# df_inter_for_ds["date_w"] = convert_to_datetime (df_inter_for_ds["last_watch_dt"])


1594786


In [289]:
# # делим на трейн и тест(валидация)
# df_inter_for_ds.sort_values(by = "date_w",  ascending=False)
# df_inter_for_ds.drop(columns=["last_watch_dt"],inplace=True)

# days = 7
# last_date = df_inter_for_ds["date_w"].max()

# df_interactions_train = df_inter_for_ds[df_inter_for_ds["date_w"] < last_date - timedelta(days=days)]

# df_interactions_test = df_inter_for_ds[df_inter_for_ds["date_w"] >= last_date - timedelta(days=days)]

In [290]:
print(df_interactions['last_watch_dt'].min(), df_interactions['last_watch_dt'].max())

2021-03-13 2021-08-22


In [291]:
# monthly = df_interactions.groupby(pd.Grouper(key='last_watch_dt', freq='M')).agg({
#     'user_id': 'count',
#     'total_dur': 'sum'
# }).rename(columns={'user_id': 'total_interactions'})
# monthly

In [292]:
old_item = df_items[df_items['release_year'] > 1960.0] # уберем старые фильмы до 1960 года
df_interactions = df_interactions[df_interactions['item_id'].isin(old_item['item_id'])]
df_interactions = df_interactions.drop(['total_dur'], axis=1, inplace=False) # уберем время просмотра

In [293]:
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'])

In [294]:
size_test = 1
test_size_days=10
df_interactions['watched_pct'] = (df_interactions['watched_pct'] >= 60).astype(int)
df_items = df_interactions['item_id'].value_counts()
active_items = df_items[df_items > 6].index


In [295]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
# max_date = df_interactions['last_watch_dt'].max()
# test_start = max_date - timedelta(days=test_size_days)
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < df_interactions['last_watch_dt'].max() - timedelta(weeks=size_test)]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= df_interactions['last_watch_dt'].max() - timedelta(weeks=size_test)]

In [296]:
# df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
# df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [297]:
df_inter.columns

Index(['user_id', 'item_id', 'last_watch_dt', 'watched_pct'], dtype='object')

In [298]:
# не обрабатываем холодных пользователей
df_interactions_test = df_interactions_test.loc[df_interactions_test["user_id"].isin(df_interactions_train["user_id"]) & df_interactions_test["item_id"].isin(df_interactions_train["item_id"])]

In [299]:
train_sorted = df_interactions_train.sort_values(['user_id', 'last_watch_dt'])
    
train = train_sorted.groupby('user_id').tail(15) # у пользователей в train берем последние 15 фильмов

In [300]:
user_interaction_test = df_interactions_test['user_id'].value_counts()
active_users_test = user_interaction_test[user_interaction_test > 1].index
test_active = df_interactions_test[df_interactions_test['user_id'].isin(active_users_test)]

user_interaction_train = train['user_id'].value_counts()
active_users_train = user_interaction_train[user_interaction_train > 1].index
train_active = train[train['user_id'].isin(active_users_train)]

test_users = test_active['user_id'].unique()
train_users = train_active['user_id'].unique()

data_train = train_active[train_active['user_id'].isin(test_users)]
data_test = test_active[test_active['user_id'].isin(train_users)]

In [301]:
test_users = set(data_test['user_id'].unique())
train_users = set(data_train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 13605
Пользователей в train: 13605
Общих пользователей: 13605


In [302]:
data_test_group = df_interactions_test.groupby(df_interactions_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,30,[8584]
1,55,[14]
2,56,[7102]
3,120,[10440]
4,136,[5294]
...,...,...
47611,1097459,"[11844, 7793]"
47612,1097470,"[1819, 6470, 6892, 12837, 6774, 13865]"
47613,1097479,[9563]
47614,1097508,[10986]


In [303]:
df_users[['user_id',"age","income"]]

,user_id,age,income
0,973171,age_25_34,income_60_90
1,962099,age_18_24,income_20_40
2,1047345,age_45_54,income_40_60
3,721985,age_45_54,income_20_40
4,704055,age_35_44,income_60_90
...,...,...,...
840191,365945,age_25_34,income_20_40
840192,339025,age_65_inf,income_0_20
840193,983617,age_18_24,income_20_40
840194,251008,NaN,NaN


In [304]:
# ratings = pd.DataFrame()
# ratings[Columns.User] = df_interactions_train['user_id']
# ratings[Columns.Item] = df_interactions_train['item_id']
# ratings[Columns.Weight] = df_interactions_train['watched_pct']
# ratings[Columns.Datetime] = df_interactions_train['last_watch_dt']

In [305]:
from sklearn.preprocessing import LabelEncoder

# Кодируем категориальные признаки (но можно их закодировать прямо при построении датасета)
le_age = LabelEncoder()
le_inc = LabelEncoder()
le_sex = LabelEncoder()
df_users["age_en"] = le_age.fit_transform(df_users["age"])
df_users["income_en"] = le_inc.fit_transform(df_users["income"])
df_users["sex_en"] = le_inc.fit_transform(df_users["sex"])
df_users.head()

,user_id,age,income,sex,kids_flg,age_en,income_en,sex_en
0,973171,age_25_34,income_60_90,М,1,1,4,1
1,962099,age_18_24,income_20_40,М,0,0,2,1
2,1047345,age_45_54,income_40_60,Ж,0,3,3,0
3,721985,age_45_54,income_20_40,Ж,0,3,2,0
4,704055,age_35_44,income_60_90,Ж,0,2,4,0


In [306]:
df_interactions_train.head()

,user_id,item_id,last_watch_dt,watched_pct
0,176549,9506,2021-05-11,1
1,699317,1659,2021-05-29,1
2,656683,7107,2021-05-09,0
3,864613,7638,2021-07-05,1
4,964868,9506,2021-04-30,1


In [307]:
df_interactions_test.head()

,user_id,item_id,last_watch_dt,watched_pct
64,73446,14488,2021-08-19,1
84,10010,512,2021-08-15,0
116,633840,14488,2021-08-18,0
141,626036,11109,2021-08-22,0
181,380725,16228,2021-08-20,0


In [308]:
df_interactions_test.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)


df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item, 
                                'last_watch_dt': Columns.Datetime, 'watched_pct': Columns.Weight},inplace=True)

C:\Users\kanze\AppData\Local\Temp\ipykernel_10860\1242639805.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_interactions_train.rename(columns={'user_id': Columns.User, 'item_id': Columns.Item,


In [309]:
def precision(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # флаги: какие рекомендованные товары действительно куплены
    flags = np.isin(recommended, bought)

    return flags.sum() / len(recommended)


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(recommended, bought)

    return flags.sum() / k


def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices = np.array(prices_recommended[:k])

    # флаги: куплен ли товар
    flags = np.isin(recommended, bought)

    # учитываем деньги
    return np.dot(flags, prices) / prices.sum()

In [310]:
def recall(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # какие купленные товары были среди рекомендованных
    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def recall_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(bought, recommended)

    return flags.sum() / len(bought)


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)
    prices_bought = np.array(prices_bought)

    # флаги: купленный товар есть в топ-k рекомендациях
    flags = np.isin(bought, recommended)

    # учитываем деньги (важны цены купленных товаров)
    return np.dot(flags, prices_bought) / prices_bought.sum()

In [311]:
def ap_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    # релевантность: рекомендованный товар куплен или нет
    flags = np.isin(recommended, bought)

    # если нет ни одного релевантного — AP = 0
    if flags.sum() == 0:
        return 0.0

    sum_precision = 0.0

    for i in range(k):
        if flags[i]:
            # precision@i+1 (на префиксе)
            precision_i = np.isin(recommended[:i+1], bought).sum() / (i + 1)
            sum_precision += precision_i

    # нормируем на число релевантных объектов в топ-k
    return sum_precision / flags.sum()


def map_k(recommended_list, bought_list, k=5, u=1):
    
    # your_code
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [312]:
def check_nan (df):
    # Проверка, есть ли вообще NaN значения в DataFrame
    has_any_nans = df.isna().any().any()
    print(f"Есть ли NaN в DataFrame: {has_any_nans}")

    # Столбцы, содержащие хотя бы один NaN
    columns_with_nans = df.columns[df.isna().any()].tolist()
    print(f"Столбцы с NaN: {columns_with_nans}")

    # Количество не-NaN значений
    non_nan_count = df.count()  # По столбцам
    print("Не-NaN значений по столбцам:")
    print(non_nan_count)
    

In [313]:
check_nan (df_interactions_train)

# Удаляем все строки, где есть хотя бы один NaN
df_interactions_train = df_interactions_train.dropna()

check_nan (df_interactions_train)

Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     1445882
item_id     1445882
datetime    1445882
weight      1445882
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     1445882
item_id     1445882
datetime    1445882
weight      1445882
dtype: int64


In [314]:
check_nan (df_interactions_test)

# Удаляем все строки, где есть хотя бы один NaN
df_interactions_test = df_interactions_test.dropna()

check_nan (df_interactions_test)

Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     80987
item_id     80987
datetime    80987
weight      80987
dtype: int64
Есть ли NaN в DataFrame: False
Столбцы с NaN: []
Не-NaN значений по столбцам:
user_id     80987
item_id     80987
datetime    80987
weight      80987
dtype: int64


In [315]:
# #user_features_df= df_users[["user_id","income","age","sex"]]

 

dc = {
    Columns.User: [],  # Используем константу Columns.User везде
    "feature": [],
    "value": []
}

for index, row in df_users.iterrows():  
    dc[Columns.User].append(row["user_id"])
    dc["feature"].append("income")
    dc["value"].append(row["income_en"])

    dc[Columns.User].append(row["user_id"])  # Исправлено!
    dc["feature"].append("age")
    dc["value"].append(row["age_en"])

    dc[Columns.User].append(row["user_id"])  # Исправлено!
    dc["feature"].append("sex")
    dc["value"].append(row["sex_en"])

user_features_df = pd.DataFrame(dc)

In [316]:
# from rectools import  dataset
# # Создаем Dataset из rectools
# dataset = dataset.Dataset.construct(ratings)

# users_uniq = ratings[Columns.User].unique()

In [317]:
from rectools import  dataset 
# Prepare a dataset to build a model
rating = dataset.Dataset.construct(interactions_df = df_interactions_train, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [318]:
from rectools.models import LightFMWrapperModel
from lightfm import LightFM

model = LightFMWrapperModel(
        # внутри модели указываем параметр no_components
        # это размезность эмбеддингов, которые выучит модель
        model=LightFM(no_components = 10,  random_state=42),
        )


In [319]:
# from rectools import  dataset
# # Создаем Dataset из rectools
# dataset = dataset.Dataset.construct(ratings)

# users_uniq = ratings[Columns.User].unique()

In [320]:
rating_val = dataset.Dataset.construct(interactions_df = df_interactions_test, 
                                   user_features_df=user_features_df, 
                                   #item_features_df=df_items
                                   make_dense_user_features=False
                                   )

In [321]:
# users_unique = df_inter_for_test_cleaned[Columns.User].unique()
# model.fit(rating_val)
# recos = model.recommend(
#     users=users_unique,
#     dataset=rating_val,
#     k= 10,
#     filter_viewed= True
   
# )

In [322]:
users_uniq = df_interactions_test[Columns.User].unique()

In [323]:
# users_unique = df_inter_for_test_cleaned[Columns.User].unique()



In [324]:
model.fit(rating_val)

In [325]:
recom = model.recommend(
    users=users_uniq,
    dataset=rating_val,
    k= 10,
    filter_viewed= True
   
)

In [326]:
recom

,user_id,item_id,score,rank
0,73446,15297,10.398910,1
1,73446,9728,10.358962,2
2,73446,10440,10.276689,3
3,73446,13865,10.207575,4
4,73446,3734,10.205492,5
...,...,...,...,...
476155,110818,512,8.409411,6
476156,110818,4151,8.346255,7
476157,110818,14488,8.246149,8
476158,110818,7571,8.215714,9


In [327]:
df_recom = recom.groupby(Columns.User)[Columns.Item].unique().reset_index()

In [328]:
df_recom.columns = Columns.UserItem

In [329]:
df_recom

,user_id,item_id
0,30,"[15297, 9728, 10440, 13865, 3734, 512, 4151, 1..."
1,55,"[15297, 9728, 10440, 3734, 13865, 512, 4151, 1..."
2,56,"[15297, 9728, 10440, 3734, 13865, 512, 4151, 1..."
3,120,"[15297, 9728, 3734, 13865, 512, 4151, 14488, 7..."
4,136,"[15297, 9728, 10440, 13865, 3734, 512, 4151, 1..."
...,...,...
47611,1097459,"[15297, 9728, 10440, 3734, 13865, 512, 4151, 1..."
47612,1097470,"[15297, 9728, 10440, 3734, 512, 4151, 14488, 7..."
47613,1097479,"[15297, 9728, 10440, 13865, 3734, 512, 4151, 1..."
47614,1097508,"[15297, 9728, 10440, 13865, 3734, 512, 4151, 1..."


In [330]:
# Проверяем структуру
print("Тип df_recom:", type(df_recom))
print("Форма df_recom:", df_recom.shape)
print("\nПервые 3 строки:")
print(df_recom.head(3))
print("\nТип данных в первой ячейке:", type(df_recom.iloc[0, 0]))
print("Содержимое первой ячейки:", df_recom.iloc[0, 0])
print("Длина первой ячейки:", len(df_recom.iloc[0, 0]) if hasattr(df_recom.iloc[0, 0], '__len__') else "не имеет длины")

Тип df_recom: <class 'pandas.core.frame.DataFrame'>
Форма df_recom: (47616, 2)

Первые 3 строки:
   user_id                                            item_id
0       30  [15297, 9728, 10440, 13865, 3734, 512, 4151, 1...
1       55  [15297, 9728, 10440, 3734, 13865, 512, 4151, 1...
2       56  [15297, 9728, 10440, 3734, 13865, 512, 4151, 1...

Тип данных в первой ячейке: <class 'numpy.int64'>
Содержимое первой ячейки: 30
Длина первой ячейки: не имеет длины


In [331]:
# Создаем словарь для маппинга item_id в названия
item_id_to_title = df_items.set_index('item_id')['title'].to_dict()

# Функция для преобразования списка ID в названия
def map_ids_to_titles(item_ids):
    """Преобразует список ID в список названий"""
    # Проверяем, что item_ids - это список или строка, которую нужно преобразовать
    if isinstance(item_ids, str):
        # Если это строка вида "[15297, 9728, ...]", преобразуем в список
        import ast
        try:
            item_ids = ast.literal_eval(item_ids)
        except:
            # Если не получается, пробуем очистить от скобок и разделить
            item_ids = item_ids.strip('[]').split(', ')
            item_ids = [int(x) for x in item_ids if x]
    
    # Преобразуем ID в названия
    return [item_id_to_title.get(item_id, f"Unknown (ID: {item_id})") for item_id in item_ids]

# Применяем функцию к колонке item_id
df_recom_with_titles = df_recom.copy()
df_recom_with_titles['item_titles'] = df_recom['item_id'].apply(map_ids_to_titles)

# Выводим результат с названиями фильмов
print("Рекомендации с названиями фильмов:")
for idx, row in df_recom_with_titles.head(100).iterrows():
    user_id = row['user_id']
    movie_titles = row['item_titles']
    print(f"Пользователь {user_id}:")
    for i, title in enumerate(movie_titles[:10], 1):
        print(f"  {i}. {title}")
    print()

AttributeError: 'Series' object has no attribute 'set_index'

In [ ]:
# # Создаем словарь для маппинга item_id в названия
# item_id_to_title = df_items.set_index('item_id')['title'].to_dict()

# # Функция для преобразования списка ID в названия
# def map_ids_to_titles(item_ids, mapping_dict):
#     return [mapping_dict.get(item_id, f"Unknown (ID: {item_id})") for item_id in item_ids]

# # Применяем маппинг к рекомендациям
# df_recom_with_titles = df_recom.apply(lambda row: [item_id_to_title.get(x, f"Unknown (ID: {x})") for x in row], axis=1)

# # Выводим результат с названиями фильмов
# print("Рекомендации с названиями фильмов:")
# for user_id, movie_titles in df_recom_with_titles.head(100).items():
#     print(f"Пользователь {user_id}:")
#     for i, title in enumerate(movie_titles, 1):
#         print(f"  {i}. {title}")
#     print()

In [ ]:
print('map_k =', map_k(df_recom['item_id'], data_test_group['item_id'], k=10, u=len(data_test_group)))

NameError: name 'result_recom' is not defined

In [ ]:
# map10 = mapk(df_recom, predicted, k=10)
# map10

In [ ]:

actual_dict = df_interactions_test.groupby(Columns.User)[Columns.Item].apply(list).to_dict()

predicted_dict = df_recom.to_dict()

mapk10_score = mapk(actual_dict, predicted_dict, k=10)


In [ ]:
mapk10_score

0.0

In [ ]:
# import pandas as pd
# from sklearn.metrics import recall_score

# d = {"user_id": [], "recos": [], 'actual': []} 
# recall_scores = []

# for u in users_unique:
#     d["user_id"].append(u)
    
#     # Рекомендации
#     r = (recos[Columns.Item][recos[Columns.User] == u]).tolist()
#     d["recos"].append(r)
    
#     # Фактические взаимодействия
#     a = (df_inter_for_test_cleaned[Columns.Item][df_inter_for_test_cleaned[Columns.User] == u]).tolist()
#     d["actual"].append(a)

#     print(f"user {u}:\n rec {r} \n act {a}")

#     # Вычисление recall для текущего пользователя
#     if len(a) > 0:  # Проверяем, что есть фактические элементы
#         # Создаем бинарные векторы для вычисления recall
#         all_items = list(set(r + a))
#         y_true = [1 if item in a else 0 for item in all_items]
#         y_pred = [1 if item in r else 0 for item in all_items]
        
#         user_recall = recall_score(y_true, y_pred, zero_division=0)
#         recall_scores.append(user_recall)
#     else:
#         recall_scores.append(0)  # Если нет фактических элементов, recall = 0

# # Средний recall по всем пользователям
# mean_recall = np.mean(recall_scores) if recall_scores else 0
# print(f"Mean Recall: {mean_recall}")

In [ ]:
# import metrics

# d = {"user_id":[], "recos": [], 'actual':[]} 
# recall = 0

# for u in users_unique:
#     d["user_id"].append(u)
#     r = (recos[Columns.Item][recos[Columns.User] == u]).to_list()
#     d["recos"].append(r)
#     a = (df_inter_for_test_cleaned[Columns.Item][df_inter_for_test_cleaned[Columns.User] == u]).to_list()
#     d["actual"].append(a)

#     print(f"user {u}:\n rec {r} \n act {a}")

#     recall += metrics.recall(r,a)

# recall = recall/len(users_unique)

# print(recall)